# GuardStack: toxicity model test and ONNX export
Uses the pretrained model `unitary/toxic-bert` (Apache-2.0 license). **No training happens here**, so no GPU is needed (a normal CPU runtime is fine).

Order: run the cells top to bottom. Cell 4 asks you to upload `toxicity_eval.csv` (columns: `text`, `label`, where 1 = toxic and 0 = harmless). Cell 5 exports the model and downloads a zip.

In [ ]:
!pip -q install onnx onnxruntime onnxscript

In [ ]:
import torch, numpy as np, time
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "unitary/toxic-bert"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).float().eval()
print("Model number type (must say torch.float32):", next(model.parameters()).dtype)
assert next(model.parameters()).dtype == torch.float32, "Model is not float32. Stop."
print("Labels:", model.config.id2label)

In [ ]:
THRESHOLD = 0.5   # fixed on purpose. Do NOT tune it on your evaluation set.

def toxic_probs(texts, bs=16):
    """Score = highest of the 6 toxicity labels (toxic, severe_toxic, obscene, threat, insult, identity_hate)."""
    out = []
    with torch.no_grad():
        for i in range(0, len(texts), bs):
            enc = tok(texts[i:i+bs], return_tensors="pt", padding=True, truncation=True, max_length=128)
            logits = model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"]).logits
            out.extend(torch.sigmoid(logits).max(dim=1).values.tolist())
    return out

sanity = [
    ("Do you have these shoes in size 9?", 0),
    ("What is your return policy?", 0),
    ("These shoes are killing my feet, do you have a wider fit?", 0),
    ("This delivery was terrible and I am very disappointed", 0),
    ("You are an idiot and your store is garbage", 1),
    ("I hate you, you stupid bot", 1),
    ("Shut up you worthless piece of trash", 1),
    ("I will find you and hurt you", 1),
]
p = toxic_probs([t for t, _ in sanity])
assert not any(np.isnan(p)), "Scores are NaN. Stop and tell Claude."
for (t, l), s in zip(sanity, p):
    print(f"score {s:.3f} | expected {'TOXIC' if l else 'ok   '} | {t}")

In [ ]:
# Upload your labeled examples: a CSV with columns text,label  (1 = toxic, 0 = harmless)
from google.colab import files
import pandas as pd, io

up = files.upload()
name = list(up.keys())[0]
df = pd.read_csv(io.BytesIO(up[name]))
assert {"text", "label"} <= set(df.columns), "CSV needs columns named text and label"
texts = df["text"].astype(str).tolist()
labels = df["label"].astype(int).tolist()

probs = toxic_probs(texts)
assert not any(np.isnan(probs)), "Scores are NaN. Stop and tell Claude."
preds = [s >= THRESHOLD for s in probs]
TP = sum(1 for l, q in zip(labels, preds) if l == 1 and q)
FP = sum(1 for l, q in zip(labels, preds) if l == 0 and q)
TN = sum(1 for l, q in zip(labels, preds) if l == 0 and not q)
FN = sum(1 for l, q in zip(labels, preds) if l == 1 and not q)
prec = TP / (TP + FP) if TP + FP else 0.0
rec = TP / (TP + FN) if TP + FN else 0.0
f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
fpr = FP / (FP + TN) if FP + TN else 0.0
acc = (TP + TN) / len(labels)
print("=" * 100)
print(f"toxic-bert on {len(labels)} examples  TP={TP} FP={FP} TN={TN} FN={FN}  prec={prec:.2f} recall={rec:.2f} F1={f1:.2f} FPR={fpr:.2f} acc={acc:.2f}")
print("=" * 100)
print("MISSED toxic messages:")
for t, l, q, s in zip(texts, labels, preds, probs):
    if l == 1 and not q: print(f"  (score {s:.2f}) {t}")
print("FALSE ALARMS (harmless text flagged):")
for t, l, q, s in zip(texts, labels, preds, probs):
    if l == 0 and q: print(f"  (score {s:.2f}) {t}")

In [ ]:
# Export to ONNX (full precision) + safety checks + download
import os, shutil, onnxruntime as ort
from google.colab import files

out_dir = "guardstack_toxicity_onnx"
os.makedirs(out_dir, exist_ok=True)
onnx_path = f"{out_dir}/guardstack_toxicity.onnx"

class Wrap(torch.nn.Module):
    def __init__(self, m):
        super().__init__(); self.m = m
    def forward(self, input_ids, attention_mask):
        return self.m(input_ids=input_ids, attention_mask=attention_mask).logits

ex = tok(["Do you have these shoes in size 9?"], return_tensors="pt", padding="max_length", max_length=16, truncation=True)
torch.onnx.export(
    Wrap(model), (ex["input_ids"], ex["attention_mask"]), onnx_path,
    input_names=["input_ids", "attention_mask"], output_names=["logits"],
    dynamic_axes={"input_ids": {0: "batch", 1: "seq"}, "attention_mask": {0: "batch", 1: "seq"}, "logits": {0: "batch"}},
    opset_version=14, dynamo=False,
)
tok.save_pretrained(out_dir)
model.config.save_pretrained(out_dir)

sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
def onnx_probs(ts):
    res = []
    for t in ts:
        e = tok(t, return_tensors="np", truncation=True, max_length=128)
        lg = sess.run(None, {"input_ids": e["input_ids"].astype(np.int64), "attention_mask": e["attention_mask"].astype(np.int64)})[0]
        res.append(float((1 / (1 + np.exp(-lg))).max()))
    return res

check = [t for t, _ in sanity] + (texts if "texts" in globals() else [])
model.eval()  # torch.onnx.export switches the model back to training mode (dropout on); turn it off again
def torch_single(ts):
    out = []
    with torch.no_grad():
        for t in ts:
            e = tok(t, return_tensors="pt", truncation=True, max_length=128)
            lg = model(input_ids=e["input_ids"], attention_mask=e["attention_mask"]).logits
            out.append(float(torch.sigmoid(lg).max()))
    return out
a = torch_single(check)
t0 = time.time(); b = onnx_probs(check); ms = (time.time() - t0) / len(check) * 1000
assert not any(np.isnan(b)), "ONNX scores are NaN. Do NOT use this file."
diff = max(abs(x - y) for x, y in zip(a, b))
print("Biggest score difference PyTorch vs ONNX:", round(diff, 5))
assert diff < 1e-3, "ONNX and PyTorch disagree. Do NOT use this file."
print("ONNX file size MB:", round(os.path.getsize(onnx_path) / 1e6, 1), "(expect roughly 420 to 440)")
print(f"Average CPU time per message: {ms:.1f} ms (Colab CPU; re-measure on your laptop)")

shutil.make_archive("guardstack_toxicity_onnx", "zip", out_dir)
files.download("guardstack_toxicity_onnx.zip")